SQLNOIR Paired Sessions
Group Name:PPG_2  
Students:Andrew Laboy & Prabhjot Kaur 
------------------------------------------------------------------------------------------------------
# Session 1
------------------------------------------------------------------------------------------------------
Date: 03-20-2026
Duration: 45 min 
Navigator (Stakeholder): Prabhjot Kaur 
Driver (Programmer): Andrew Laboy

Business Problem #1:

The company is gaining many customers but, it is unclear how many return after their first purchase. The goal is to identify one time versus repeat customers and compare the revenue generated by each group.

- Requirement 1: Count the number of orders per customer.
- Requirement 2: Classify customers as one time or repeat buyers.
- Requirement 3: Calculate total revenue for each group.
------------------------------------------------------------------------------------------------------
Query Evolution
------------------------------------------------------------------------------------------------------
ATTEMPT #1:

In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderID) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalOrders DESC

RESULT/ISSUE: He didnt seperate them into groups like one time buyer and the repeated customer. 
Dont list the customers individually I want the groups side by side.

FINAL QUERY:


In [ ]:
USE Northwinds2024Student;
GO
WITH CustomerOrders AS (
    SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderID) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
    FROM Sales.Customer c
    JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
    JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
    GROUP BY c.CustomerId, c.CustomerCompanyName
),
CustomerSegment AS (
    SELECT *,
        CASE 
        WHEN TotalOrders = 1 THEN 'One-Time Buyer'
        ELSE 'Repeat Customer'
        END AS CustomerType
    FROM CustomerOrders
)
SELECT 
    CustomerType,
    COUNT(CustomerId) AS TotalCustomers,
    SUM(TotalOrders) AS TotalOrders,
    SUM(TotalRevenue) AS TotalRevenue,
    AVG(TotalRevenue) AS AvgRevenuePerCustomer
FROM CustomerSegment
GROUP BY CustomerType

FINAL RESULTS: Based off the query we can see since there are no one time buyers on   
record our retention is really strong. We should focus on getting new customers because of that.

------------------------------------------------------------------------------------------------------
# Session 2
------------------------------------------------------------------------------------------------------
Date: 03-20-202
Duration: 1 Hour 15 min
Navigator (Stakeholder): Andrew Laboy
Driver (Programmer): Prabhjot Kaur

Business Problem # 2:
Were starting to see inconsistency in how quickly orders are being shipped out. I need 
you to find out which shipping companies are taking the longest on average to deliver 
orders and if that changes depending on the destination country.

Requirements:
- Requirement 1: Pull orders and calculate delivery time.
- Requirement 2: Make sure to break it down by destination country so we can see any varity.
- Requirement 3: Note any shipper whose average delivery time is higher than others.
------------------------------------------------------------------------------------------------------
Query Evolution
------------------------------------------------------------------------------------------------------
ATTEMPT #1:


In [ ]:
SELECT 
    ShipperId,
    AVG(DATEDIFF(DAY, OrderDate, ShipToDate)) AS AvgDays
FROM Sales.[Order]
GROUP BY ShipperId;


RESULT/ISSUE:
This query calculated the average delivery time but, it only used the shipper ID. 
It did not include the actual shipper names or the destination country so the results were not very useful.

ATTEMPT #2:

In [ ]:
SELECT 
    s.ShipperCompanyName,
    o.ShipToCountry,
    AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS AvgDays
FROM Sales.[Order] o
JOIN Sales.Shipper s
    ON o.ShipperId = s.ShipperID
GROUP BY 
    s.ShipperCompanyName,
    o.ShipToCountry;

RESULT/ISSUE:

This version was better because it included the shipper name and country but
it still counted orders where the shipped date was NULL, which could make the averages inaccurate.

FINAL QUERY:

In [ ]:
USE Northwinds2024Student;
GO

SELECT 
    s.ShipperCompanyName AS Shipper,
    o.ShipToCountry,
    AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS AvgDeliveryDays
FROM Sales.[Order] o
JOIN Sales.Shipper s
    ON o.ShipperId = s.ShipperID
WHERE o.ShipToDate IS NOT NULL
GROUP BY 
    s.ShipperCompanyName,
    o.ShipToCountry
ORDER BY 
    AvgDeliveryDays DESC;

FINAL RESULTS:

The final query lists each shipper with its destination country and average delivery time. 
Some shippers take longer than others, especially in certain countries.These should be reviewed and faster options can be considered.

------------------------------------------------------------------------------------------------------
Session 3
------------------------------------------------------------------------------------------------------

Date: 03-21-26 
Duration: 50 min
Navigator (Stakeholder): Prabhjot Kaur
Driver (Programmer): Andrew Laboy

Business Problem #3 

Sales patterns are unclear, making it difficult to plan promotions and staffing. 
The goal is to identify which days of the week and months generate the highest revenue.

- Requirement 1: Calculate total revenue from order data.
- Requirement 2: Group results by day of the week and by month.
- Requirement 3: Identify the highest-performing days and months.
------------------------------------------------------------------------------------------------------
Query Evolution
------------------------------------------------------------------------------------------------------
ATTEMPT #1:


In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    MONTH(o.OrderDate) AS OrderMonth,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
GROUP BY MONTH(o.OrderDate)
ORDER BY TotalRevenue DESC

RESULT/ISSUE: The month numbers are still very unreadable. Also make sure to add
day of the week.

ATTEMPT #2:

In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    DATENAME(WEEKDAY, o.OrderDate) AS DayOfWeek,
    MONTH(o.OrderDate) AS OrderMonth,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
GROUP BY DATENAME(WEEKDAY, o.OrderDate), MONTH(o.OrderDate)
ORDER BY TotalRevenue DESC


RESULT/ISSUE: Months are fixed but still hard to read since every combination is in its own row.
We need to see day of the week and month independently.

FINAL QUERY:

In [ ]:
USE Northwinds2024Student;
GO
WITH RevenueByDay AS (
    SELECT 
        DATENAME(WEEKDAY, o.OrderDate) AS DayOfWeek,
        DATEPART(WEEKDAY, o.OrderDate) AS DayNumber,
        SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
    GROUP BY DATENAME(WEEKDAY, o.OrderDate), DATEPART(WEEKDAY, o.OrderDate)
),
RevenueByMonth AS (
    SELECT 
        DATENAME(MONTH, o.OrderDate) AS OrderMonth,
        MONTH(o.OrderDate) AS MonthNumber,
        SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderID = od.OrderID
    GROUP BY DATENAME(MONTH, o.OrderDate), MONTH(o.OrderDate)
)
SELECT 'Day of Week' AS GroupType, DayOfWeek AS Period, TotalRevenue
FROM RevenueByDay
UNION ALL
SELECT 'Month', OrderMonth, TotalRevenue
FROM RevenueByMonth
ORDER BY GroupType, TotalRevenue DESC


FINAL RESULTS:
Based off final results we can see that Sunday is our highest
revenue day at $284,393.64. Our lowest days would be Monday and Tuesday. Regarding 
the months our highest paid month is April with $190,329.95. January and March are 
also strong. June out of all the months is our lowest in revenue at $39,088.00.
With this information we can try and focus on days and months where we are at our lowest.

------------------------------------------------------------------------------------------------------
Session 4
------------------------------------------------------------------------------------------------------

Date: 03-21-26
Duration: 1 Hour 10 min
Navigator (Stakeholder): Andrew Laboy 
Driver (Programmer): Prabhjot Kaur

Business Problem #4:
I want to understand which products are actually bringing in money for our revenue versus 
just having high order volume. A product can sell a lot of units but still be low in value. 
Can you break down total revenue by category and show me which ones are 
underperforming compared to how often they are ordered?

- Requirement 1: Gather all the products and calculate the total revenue.
- Requirement 2: Group those results by product category so we can see the performance.
- Requirement 3: If order volume is high but revenue low then those are the underperformers. 

------------------------------------------------------------------------------------------------------
Query Evolution
------------------------------------------------------------------------------------------------------
ATTEMPT #1:

In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    p.CategoryId,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue
FROM Sales.OrderDetail od
JOIN Production.Product p
    ON od.ProductId = p.ProductId
GROUP BY 
    p.CategoryId;

RESULT/ISSUE:

This query calculated total revenue but, it only used CategoryID and did not show actual category names. 
It also didn’t include order count, so there was no way to compare volume.

ATTEMPT #2:


In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    c.CategoryName,
    SUM(od.UnitPrice * od.Quantity) AS TotalRevenue,
    COUNT(od.OrderId) AS OrderCount
FROM Sales.OrderDetail od
JOIN Production.Product p
    ON od.ProductId = p.ProductId
JOIN Production.Category c
    ON p.CategoryId = c.CategoryId
GROUP BY 
    c.CategoryName;
    

RESULT/ISSUE:

This version improved the results by adding category names and order count but,
it still doesnt show which categories were underperforming.

FINAL QUERY:

In [ ]:
USE Northwinds2024Student;
GO

WITH CategoryStats AS (
    SELECT 
        c.CategoryName,
        SUM(od.UnitPrice * od.Quantity) AS TotalRevenue,
        COUNT(od.OrderId) AS OrderCount
    FROM Sales.OrderDetail od
    JOIN Production.Product p
        ON od.ProductId = p.ProductId
    JOIN Production.Category c
        ON p.CategoryId = c.CategoryId
    GROUP BY 
        c.CategoryName
)
SELECT 
    CategoryName,
    TotalRevenue,
    OrderCount,
    CASE 
        WHEN OrderCount > 100 AND TotalRevenue < 50000 THEN 'Underperforming'
        ELSE 'Performing'
    END AS PerformanceStatus
FROM CategoryStats
ORDER BY 
    TotalRevenue DESC;

FINAL RESULTS:
The query displayed each product category with total revenue, number of orders, and performance status. Some categories had high 
order counts but relatively low revenue showing underperformance. The company should adjust pricing strategies or promote
higher value products within those categories.

------------------------------------------------------------------------------------------------------
# Session 5
------------------------------------------------------------------------------------------------------

Date: 03-22-26 
Duration: 50 min
Navigator (Stakeholder): Prabhjot Kaur
Driver (Programmer): Andrew Laboy

Business Problem #5

A single report is needed to summarize key business insights, 
including discount impact, employee involvement, shipping delays, and revenue loss.

- Requirement 1: Identify categories with the highest discounts.
- Requirement 2: Determine employees involved in high discount transactions.
- Requirement 3: Include shipping performance and total revenue loss in one output.

------------------------------------------------------------------------------------------------------
Query Evolution
------------------------------------------------------------------------------------------------------

ATTEMPT #1:

In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    c.CategoryName,
    SUM(od.LineAmount - od.DiscountedLineAmount) AS TotalRevenueLoss
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY TotalRevenueLoss DESC

RESULT/ISSUE: Good start now just add the employee data and shipping delays   
so it can be into one single output.

ATTEMPT #2:

In [ ]:
USE Northwinds2024Student;
GO
SELECT 
    c.CategoryName,
    AVG(od.DiscountPercentage) AS AvgDiscount,
    SUM(od.LineAmount - od.DiscountedLineAmount) AS TotalRevenueLoss,
    COUNT(od.OrderId) AS TotalOrders
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY TotalRevenueLoss DESC

FINAL RESULTS:
Based off the chart we can see that beverages and dairy products are  
our highest revenue loss from discounts. Meat/Poultry have the highest average discount 
rate at 6.4% but not ordered a lot. 